In [32]:
# imports
import asyncio
import inspect
import json
import urllib
import base64
import os
import tempfile
import gradio as gr
from openai import OpenAI
import sqlite3
import edge_tts
from io import BytesIO
from PIL import Image

In [2]:
# Initialization

MODEL = "llama3.2:latest"
IMAGE_MODEL = "x/flux2-klein:4b"
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [3]:
!ollama pull x/flux2-klein:4b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
Error: this model requires MLX support, but the MLX runtime is not available


In [4]:
# Run this inside your Jupyter Notebook cell:
!uv pip install edge-tts pillow

Using Python 3.12.12 environment at: C:\practice_evt\llm_course_by_donner\llm_engineering\.venv
Checked 2 packages in 952ms


In [5]:

def init_db():
    conn = sqlite3.connect("art_quotes.db")
    cursor = conn.cursor()

    # Create table if it doesn't exist
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS quotes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            artist TEXT NOT NULL,
            era TEXT,
            quote TEXT NOT NULL UNIQUE
        )
    """)

    # Seed with initial data
    sample_data = [
        (
            "Vincent van Gogh",
            "Post-Impressionism",
            "I dream my painting and I paint my dream.",
        ),
        (
            "Leonardo da Vinci",
            "High Renaissance",
            "Learning never exhausts the mind.",
        ),
        (
            "Leonardo da Vinci",
            "High Renaissance",
            "Painting is poetry that is seen rather than felt.",
        ),
        (
            "Pablo Picasso",
            "Cubism",
            "Everything you can imagine is real.",
        ),
        (
            "Claude Monet",
            "Impressionism",
            "I would like to paint the way a bird sings.",
        ),
    ]

    cursor.executemany(
        """
        INSERT OR IGNORE INTO quotes (artist, era, quote) VALUES (?, ?, ?)
    """,
        sample_data,
    )

    conn.commit()
    conn.close()
    print("Database initialized successfully!")



In [6]:
init_db()

Database initialized successfully!


In [7]:
def lookup_quote(quote_keywords: str) -> str:
    """Searches the SQLite database for a quote matching the keywords."""
    conn = sqlite3.connect("art_quotes.db")
    cursor = conn.cursor()

    # Search quote text using LIKE operator
    query_pattern = f"%{quote_keywords}%"
    cursor.execute(
        """
        SELECT artist, era, quote FROM quotes 
        WHERE quote LIKE ? OR artist LIKE ?
        LIMIT 1
    """,
        (query_pattern, query_pattern),
    )

    row = cursor.fetchone()
    conn.close()

    if row:
        return json.dumps(
            {"found": True, "artist": row[0], "era": row[1], "quote": row[2]}
        )
    else:
        return json.dumps(
            {
                "found": False,
                "message": f"No artist quote found matching '{quote_keywords}'",
            }
        )


def add_artist_quote(artist: str, quote: str, era: str = "Unknown") -> str:
    """Inserts a new quote and artist into the SQLite database."""
    conn = sqlite3.connect("art_quotes.db")
    cursor = conn.cursor()

    try:
        cursor.execute(
            """
            INSERT INTO quotes (artist, era, quote) VALUES (?, ?, ?)
        """,
            (artist, era, quote),
        )
        conn.commit()
        result = json.dumps(
            {
                "success": True,
                "message": f"Successfully added quote for '{artist}' to the database.",
            }
        )
    except sqlite3.IntegrityError:
        result = json.dumps(
            {
                "success": False,
                "message": "This exact quote already exists in the database.",
            }
        )
    except Exception as e:
        result = json.dumps({"success": False, "message": str(e)})

    conn.close()
    return result

In [36]:
# ==========================================
# 2. Hardened Tool Schemas for Ollama / Llama3.2
# ==========================================
tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_quote",
            "description": "Searches database to identify the artist and context behind a quote",
            "parameters": {
                "type": "object",
                "properties": {
                    "quote_keywords": {
                        "type": "string", 
                        "description": "Keywords or phrase from the quote"
                    }
                },
                "required": ["quote_keywords"],
                "additionalProperties": False
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "add_artist_quote",
            "description": "Saves a new artist name and quote pair into the database",
            "parameters": {
                "type": "object",
                "properties": {
                    "artist": {
                        "type": "string", 
                        "description": "Name of the artist"
                    },
                    "quote": {
                        "type": "string", 
                        "description": "The quote stated by the artist"
                    },
                    "era": {
                        "type": "string", 
                        "description": "Artistic era/movement, defaults to Unknown if not specified"
                    },
                },
                "required": ["artist", "quote", "era"],  # Requiring era prevents trailing bracket drops
                "additionalProperties": False
            },
        },
    },
]

In [33]:
# ==========================================
# 3. Local Multimodal Helpers (Artist & Talker)
# ==========================================


def generate_artist_image(artist_name: str):
    """Generates a 2D animated portrait using Ollama native endpoint with an online visual fallback."""
    # 1. Stylized prompt enforcing low-complexity 2D animation
    prompt = (
        f"2D animated vector illustration portrait of {artist_name}, "
        f"studio Ghibli anime style, flat colors, bold outlines, vibrant art background"
    )

    # 2. Primary Method: Native Ollama REST API Call
    try:
        url = "http://localhost:11434/api/generate"
        payload = json.dumps({
            "model": IMAGE_MODEL,
            "prompt": prompt,
            "stream": False,
        }).encode("utf-8")

        req = urllib.request.Request(
            url, data=payload, headers={"Content-Type": "application/json"}
        )

        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode("utf-8"))

            # Check for base64 output string in response payload
            if "response" in data and data["response"]:
                raw_b64 = data["response"].strip()
                if raw_b64.startswith("data:image"):
                    raw_b64 = raw_b64.split(",")[1]
                return Image.open(BytesIO(base64.b64decode(raw_b64)))

    except Exception as e:
        print(f"Local Ollama image generation skipped: {e}")

    # 3. Secondary Fallback: Free zero-token animated image generator
    try:
        encoded_prompt = urllib.parse.quote(prompt)
        fallback_url = (
            f"https://image.pollinations.ai/prompt/{encoded_prompt}?width=512&height=512&nologo=true"
        )
        req = urllib.request.Request(
            fallback_url, headers={"User-Agent": "Mozilla/5.0"}
        )
        with urllib.request.urlopen(req, timeout=10) as resp:
            return Image.open(BytesIO(resp.read()))
    except Exception as e:
        print(f"Fallback fetch failed: {e}")

    # 4. Final Fallback: Blank dark canvas
    return Image.new("RGB", (512, 512), color=(40, 44, 52))

def generate_speech(message: str):
    """Generates an MP3 file path from text using edge-tts."""
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    
    async def _generate():
        communicate = edge_tts.Communicate(message, "en-US-ChristopherNeural")
        await communicate.save(temp_file.name)

    asyncio.run(_generate())
    return temp_file.name

In [28]:
# ==========================================
# 4. Tool Execution & Core Chat Loop
# ==========================================


def dynamic_tool_dispatcher(tool_calls, available_functions):
    tool_responses = []
    executed_results = []

    for tool_call in tool_calls:
        fn_name = tool_call.function.name
        raw_args = tool_call.function.arguments

        # Safely attempt JSON parsing
        try:
            if isinstance(raw_args, str):
                args = json.loads(raw_args)
            else:
                args = raw_args
        except json.JSONDecodeError as e:
            # Catch malformed JSON generated by the model gracefully
            print(f"Tool call JSON syntax error from model: {raw_args}")
            tool_responses.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps({
                    "error": "Invalid arguments JSON provided by model. Retry with full parameters."
                }),
            })
            continue

        if fn_name in available_functions:
            target_func = available_functions[fn_name]
            sig = inspect.signature(target_func)
            valid_args = {k: v for k, v in args.items() if k in sig.parameters}

            try:
                raw_result = target_func(**valid_args)
                parsed_result = (
                    json.loads(raw_result)
                    if isinstance(raw_result, str)
                    else raw_result
                )

                tool_responses.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": (
                        raw_result
                        if isinstance(raw_result, str)
                        else json.dumps(raw_result)
                    ),
                })
                executed_results.append({
                    "name": fn_name,
                    "result": parsed_result,
                })
            except Exception as e:
                tool_responses.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps({"error": f"Execution failed: {str(e)}"}),
                })
        else:
            tool_responses.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps({"error": f"Function '{fn_name}' not registered."}),
            })

    return tool_responses, executed_results

In [23]:
TOOL_REGISTRY = {
    "lookup_quote": lookup_quote,
    "add_artist_quote": add_artist_quote,
}


def extract_artists_from_results(executed_results):
    found_artists = []
    for item in executed_results:
        res = item.get("result", {})
        if isinstance(res, dict) and "artist" in res:
            found_artists.append(res["artist"])
    return found_artists

In [37]:
system_content = """You are an Art History Assistant with access to a database of quotes.

When you need to call a function, strictly output valid JSON matching the schema.
DO NOT use custom formatting like 'parameters{'.

Examples of valid tool execution syntax:
- For lookup: {"name": "lookup_quote", "arguments": {"quote_keywords": "Virginia Woolf"}}
- For adding: {"name": "add_artist_quote", "arguments": {"artist": "Salvador Dali", "quote": "Have no fear of perfection", "era": "Surrealism"}}
"""

In [38]:
def chat(history):
    formatted_messages = [{"role": "system", "content": system_content}]
    for h in history:
        formatted_messages.append({"role": h["role"], "content": h["content"]})

    response = ollama.chat.completions.create(
        model=MODEL, messages=formatted_messages, tools=tools
    )
    message = response.choices[0].message

    artists = []
    image = None

    while message.tool_calls:
        tool_responses, executed_results = dynamic_tool_dispatcher(
            message.tool_calls, TOOL_REGISTRY
        )
        artists.extend(extract_artists_from_results(executed_results))

        formatted_messages.append(message)
        formatted_messages.extend(tool_responses)

        response = ollama.chat.completions.create(
            model=MODEL, messages=formatted_messages, tools=tools
        )
        message = response.choices[0].message

    reply = message.content or "Action executed."
    history.append({"role": "assistant", "content": reply})

    voice_path = generate_speech(reply)
    if artists:
        image = generate_artist_image(artists[0])

    return history, voice_path, image

In [39]:
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

In [40]:
# ==========================================
# 5. Gradio UI
# ==========================================
with gr.Blocks() as ui:
    gr.Markdown("# 🎨 Quote-to-Artist AI (SQLite Database + Multi-tool Ollama Agent)")

    with gr.Row():
        chatbot = gr.Chatbot(height=450, type="messages")
        image_output = gr.Image(height=450, interactive=False, label="Artist Visual")
    with gr.Row():
        audio_output = gr.Audio(autoplay=True, label="Audio Response")
    with gr.Row():
        message = gr.Textbox(
            label="Ask about a quote or add a new one (e.g., 'Who said I dream my painting?'):"
        )

    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot],
    ).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Local Ollama image generation skipped: HTTP Error 404: Not Found
Local Ollama image generation skipped: HTTP Error 404: Not Found


In [20]:
ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\practice_evt\llm_course_by_donner\llm_engineering\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\practice_evt\llm_course_by_donner\llm_engineering\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\practice_evt\llm_course_by_donner\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\practice_evt\llm_course_by_donner\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\practice_ev